# 2.2 Building Custom Tools

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Learning objectives

- Turn a plain Python function into a model-callable tool with `@tool`, and explain what the decorator generates from type hints and the docstring.
- Build a `StructuredTool` with an explicit Pydantic args schema, and say when to prefer it over `@tool`.
- Write tool names and descriptions that the model actually uses to choose the right tool with the right arguments.
- Handle input validation and errors inside tools so the agent gets a useful observation instead of a crash.
- Use async tools and recognise the retriever-as-tool pattern (built properly in Module 3).

## Tools are just functions with a name, a description, and a schema

Everything you read about tools boils down to one mechanical fact: a model cannot call your Python directly. It can only emit text shaped like a function call, something like `{"name": "add", "args": {"a": 4, "b": 5}}`. Your harness parses that text, runs the real function, and feeds the result back as an observation.

A tool is the adapter between the model's text output and your code. It carries three things the model needs:

1. **A name** so the model can refer to the tool.
2. **A description** so the model knows when to use it.
3. **An argument schema** so the model knows what inputs are valid.

LangChain's `@tool` decorator builds all three from one Python function. The function name becomes the tool name, the docstring becomes the description, and the type hints become the argument schema.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together.

    Use this for any arithmetic question that needs addition.
    Do not use it for non-numeric input.
    """
    return a + b

After decoration, `add` is a `StructuredTool` object. Inspect it and you see exactly what the model will see:

In [ ]:
print(add.name)          # add
print(add.description)   # Add two numbers together. ... (the docstring)
print(add.args)          # {'a': {'type': 'number'}, 'b': {'type': 'number'}}

That third line is worth staring at. The model's entire understanding of how to call your tool comes from that dict, which came from your type hints plus the `Field(description=...)` text you may add. The code inside the function body? The model never sees it. This is the single most misunderstood thing about tools, and the single biggest source of bad agent behaviour.

## The docstring is the model's instruction manual

Imagine you are onboarding a colleague who has never seen your codebase and will only ever read one paragraph about your function. That paragraph is the docstring. When an agent picks the wrong tool, or calls the right tool with nonsense arguments, the fix is almost always in the docstring, not in the code.

A good tool description has four parts:

1. **What it does**, in one sentence. Verbs, concrete nouns.
2. **When to use it** ("Use this when...").
3. **When NOT to use it** ("Do not use this for...").
4. **Argument hints**, if an argument's format is not obvious (a date format, a unit, a currency code).

Compare:

In [ ]:
# Weak. The model guesses everything.
@tool
def lookup_order(order_id: str) -> str:
    """Looks up an order."""
    ...

# Strong. The model knows when, when-not, and what format to pass.
@tool
def lookup_order(order_id: str) -> str:
    """Look up a customer order by its ID and return status, total, and delivery date.

    Use this when the user asks about an order, a delivery, or a refund.
    Do not use this for product questions or account issues.
    Args:
        order_id: The order ID exactly as the user wrote it, e.g. "1024" or "ORD-1024".
    """
    ...

Two honest observations from production practice. First, "when not to use it" lines genuinely reduce wrong-tool calls; the model's tool-selection accuracy depends on negative guidance as much as positive. Second, examples in descriptions work: a format example like `e.g. "1024"` cuts down hallucinated argument shapes more than any amount of schema cleverness.

## Naming: snake_case verbs the model can parse

Tool names travel through the model's prompt on every loop iteration, so keep them short and unambiguous:

- Start with a verb: `lookup_order`, `calculate_tax`, `get_current_time`. Not `order` or `OrderInfo`.
- Use snake_case. Models handle it cleanly; CamelCase or spaces cause parsing hiccups in some providers' tool schemas.
- Avoid collisions with argument names, and avoid near-duplicates across your tool list (`get_weather` and `get_forecast` in the same list is asking for confusion). If two tools do similar things, merge them with a parameter instead.
- Keep it stable. If you rename a tool later, every prompt, evaluation trace, and cached decision that mentioned the old name goes stale.

## StructuredTool with an explicit Pydantic schema

For most functions, `@tool` infers everything it needs. But when you need strict control over argument validation, constraints, or descriptions per argument, write the schema explicitly with Pydantic and pass it to `@tool` (or to `StructuredTool` directly).

In [ ]:
from datetime import datetime, timezone
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class DateTimeInput(BaseModel):
    """Arguments for the get_current_time tool."""

    tz: str = Field(
        default="UTC",
        description="IANA timezone name, e.g. 'Europe/Berlin' or 'Asia/Karachi'.",
    )

@tool(args_schema=DateTimeInput)
def get_current_time(tz: str = "UTC") -> str:
    """Return the current date and time.

    Use this whenever the user asks about 'now', 'today', the current date,
    or anything time-sensitive. Do not guess the date; call this instead.
    """
    from zoneinfo import ZoneInfo
    return datetime.now(ZoneInfo(tz)).isoformat()

print(get_current_time.args)
# {'tz': {'title': 'Tz', 'description': "IANA timezone name...", 'default': 'UTC', 'type': 'string'}}

The `Field(description=...)` text lands in the JSON schema the model reads, argument by argument. That is where you put format guidance: units, date formats, allowed values. For a fixed set of values, prefer a `Literal` type: the model sees a dropdown of allowed values instead of a free-form string, which eliminates a whole class of hallucinated inputs.

The direct `StructuredTool` constructor covers cases `@tool` cannot: wrapping an existing function without touching its source, or combining a sync function with an async coroutine:

In [ ]:
from langchain_core.tools import StructuredTool

def search_catalog(query: str, max_results: int = 5) -> str:
    # some existing search implementation
    ...

search_tool = StructuredTool(
    name="search_catalog",
    description="Search the product catalog by keyword. Use when the user wants to find items to buy.",
    args_schema=CatalogInput,   # a Pydantic model you defined
    func=search_catalog,
)

Reach for the explicit schema when: arguments need formats the model would otherwise guess (dates, IDs, currency codes); you want per-argument descriptions; or you need strict constraints like `Literal` or regex patterns. For simple functions with obvious types, `@tool` inference is fine and there is no virtue in extra ceremony.

## Validation and error handling: return errors, do not raise them

Tools fail. The model passes a malformed date, the API is down, the file does not exist. What happens next decides whether the agent recovers or the whole run crashes. The rule: **a tool should return a friendly error message, not raise an exception.** The returned string becomes an observation the model reads, and a good model will retry with corrected arguments.

In [ ]:
from langchain_core.tools import tool

KNOWN_CITIES = {"london", "berlin", "tokyo", "karachi"}

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Use this when the user asks about weather or temperature.
    Args:
        city: City name in lowercase, e.g. 'london'.
    """
    if city.lower() not in KNOWN_CITIES:
        return (
            f"Error: I do not have weather data for '{city}'. "
            f"Available cities: {', '.join(sorted(KNOWN_CITIES))}. "
            "Ask the user which of these cities they meant, or answer without weather data."
        )
    # ... real lookup here ...
    return f"Weather in {city}: 18C, overcast."

Notice the error message does three things: says what went wrong, lists valid options, and tells the model what to do next. Compare that with a raised exception, which produces an empty tool result or a crashed run and teaches the model nothing.

There is a formal version of this: raise `ToolException` inside the tool and set `handle_tool_error=True` on the tool, and the framework converts the exception into an observation automatically. It works on any agent without middleware. But for the agents you will build in this course, the plain pattern above is easier to read, easier to test, and keeps the error message exactly where the logic is. One more practical note: keep tool functions pure about their inputs where you can. Validate early, fail with text, never with a stack trace.

Also validate shape, not just membership. If a tool takes a date, parse it inside the tool with `datetime.strptime` in a try/except and return the accepted format in the error message. The model will usually get it right on the second attempt.

## Async tools

If your tool calls an API, a database, or anything else with awaitable IO, make the function async and `@tool` handles the rest: the decorated tool exposes both `invoke` and `ainvoke`.

In [ ]:
import asyncio
from langchain_core.tools import tool

@tool
async def fetch_exchange_rate(base: str, target: str = "EUR") -> str:
    """Fetch the current exchange rate between two currency codes.

    Use this for currency conversion questions.
    Args:
        base: Source currency code, e.g. 'PKR', 'USD'.
        target: Target currency code, e.g. 'EUR'. Defaults to 'EUR'.
    """
    await asyncio.sleep(0.1)  # stand-in for a real HTTP call
    rates = {("USD", "EUR"): 0.92, ("PKR", "EUR"): 0.0033}
    rate = rates.get((base.upper(), target.upper()))
    if rate is None:
        return f"Error: no rate available for {base} to {target}. Supported pairs: USD->EUR, PKR->EUR."
    return f"1 {base.upper()} = {rate} {target.upper()}"

result = asyncio.run(fetch_exchange_rate.ainvoke({"base": "USD", "target": "EUR"}))
print(result)  # 1 USD = 0.92 EUR

Two things to note. First, an async tool can be `invoke`d from sync code and `ainvoke`d from async code; you do not need separate versions. Second, mixing blocking sync code inside an async tool blocks the event loop: in a server or in LangGraph's concurrent node execution, that becomes a latency bug. If the underlying library is sync-only, wrap the call with `asyncio.to_thread`.

## Preview: retriever as a tool (Module 3 does this properly)

One pattern is so common it deserves its own name: wrapping a retriever as a tool. You will build this fully in Module 3 when retrieval is covered, but the shape is worth seeing now because it is the cleanest example of a tool boundary.

In [ ]:
from langchain_core.tools import tool

@tool
def search_docs(query: str) -> str:
    """Search the internal documentation for relevant information.

    Use this when the user asks a factual question that documentation could answer.
    Args:
        query: A short search query of keywords, e.g. 'refund policy'.
    """
    # In Module 3 this calls a real vector-store retriever.
    docs = {
        "refund policy": "Refunds are issued within 14 days of purchase to the original payment method.",
        "shipping": "Standard shipping takes 3-5 business days within the EU.",
    }
    hits = [v for k, v in docs.items() if query.lower() in k.lower()]
    return "\n\n".join(hits) if hits else f"No documentation found for '{query}'. Try a shorter query."

The tool description tells the model when to search; the returned text becomes an observation the model can quote. In Module 3 the dict lookup becomes a real retriever over real documents, but the tool wrapper stays the same shape.

## Putting the pieces together

Three runnable tools that you will reuse in lesson 2.3's agent loop. They work with no network and no API key beyond your model key:

In [ ]:
import ast
import operator
from datetime import datetime, timezone
from langchain_core.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression.

    Use this for any math the user asks for instead of doing mental arithmetic.
    Args:
        expression: A Python arithmetic expression using +, -, *, /, ** and parentheses,
            e.g. '38 + 47' or '(120 * 1.19) - 50'.
    """
    allowed = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
               ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed:
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed:
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")
    try:
        return str(_eval(ast.parse(expression, mode="eval").body))
    except Exception:
        return (
            f"Error: could not evaluate '{expression}'. "
            "Send a plain arithmetic expression using +, -, *, /, ** and parentheses."
        )

@tool
def get_current_time() -> str:
    """Return the current UTC date and time as an ISO string.

    Use this whenever the user asks about 'now', 'today', or the current date.
    """
    return datetime.now(timezone.utc).isoformat()

Try them directly, no agent involved: `calculate.invoke({"expression": "38 + 47"})` returns `'85'`; `get_current_time.invoke({})` returns the timestamp. Direct invocation is also your fastest debugging tool: before blaming the agent, call the tool yourself and check its output.

## What can go wrong

- **The model ignores your tool.** Almost always a description problem: the "when to use it" sentence is missing, vague, or the tool overlaps with a sibling tool. Rewrite the description; do not add parameters.
- **Hallucinated arguments.** The model invents an argument your schema does not have, or passes a date as "next Friday". Fix with `Field(description=...)` format examples, `Literal` for fixed choices, and in-tool validation that returns the accepted format in the error message.
- **Tools with side effects run twice.** If a tool sends an email or charges a card, an agent that retries can cause real damage. Make side-effecting tools idempotent where possible, require explicit confirmation arguments, and gate them behind human-in-the-loop approval (Module 6) rather than hoping the model behaves.
- **Leaking secrets in tool output.** A tool that returns raw API responses can paste tokens or internal IDs into the model's context, where they may surface in the final answer. Strip sensitive fields before returning.
- **Slow tools blocking the loop.** A synchronous 30-second HTTP call inside a tool freezes the whole agent loop. Use async tools with real awaitable clients, or `asyncio.to_thread` for sync libraries.
- **Schema too clever.** Deeply nested Pydantic models and unions in tool args confuse most models. Keep args flat: strings, numbers, booleans, short lists. If you need structure, take a JSON string and parse it inside the tool.
- **`return_direct` confusion.** The old `return_direct=True` flag made an agent stop looping after a tool ran. With `create_agent` it still short-circuits the loop, but combined with middleware-based error handling it can swallow errors into silence. Unless you have a specific reason, leave it at the default `False` and let the model decide when it is done.

## Check your understanding

**1. The model never sees your tool's code. What three things does it see instead, and where does each come from?**
It sees the tool name (from the Python function name), the description (from the docstring), and the argument schema (from the type hints, or from the explicit Pydantic schema). The function body is invisible to it.

**2. Why does a good tool description include "Do not use this for..." guidance?**
Tool selection is a classification problem the model solves from descriptions alone. Negative guidance shrinks the decision boundary: it tells the model when *not* to pick this tool, which measurably reduces wrong-tool calls, especially when tools overlap.

**3. When should you write an explicit Pydantic args schema instead of letting `@tool` infer one?**
When arguments need formats the model would guess wrong (dates, IDs, units), when each argument needs its own description text, when you need strict constraints like `Literal` choices or regex patterns, or when wrapping an existing function without editing its source.

**4. A tool call fails because the model passed a bad argument. Should the tool raise an exception or return an error string? Why?**
Return an error string. The string becomes an observation the model reads, and if it says what was wrong and what the valid options are, the model can usually self-correct on the next loop iteration. A raised exception crashes the run or yields an empty observation that teaches the model nothing.

**5. Why keep tool argument schemas flat (strings, numbers, booleans) instead of using nested Pydantic models?**
Most models produce unreliable output against deeply nested or union-typed schemas. Flat args mean fewer chances for the model to hallucinate structure. If you genuinely need structure, accept a JSON string argument and parse and validate it inside the tool, where Python does the work instead of the model.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.